In [ ]:
import requests, json, os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path('.env'))

BASE_URL = 'http://localhost:8001'
NOTION_API_TOKEN = os.getenv('NOTION_API_TOKEN', '')
# Parse NOTION_ALLOWED_PAGE_IDS (can be JSON array or CSV)
pages_env = os.getenv('NOTION_ALLOWED_PAGE_IDS', '')
try:
    NOTION_ALLOWED_PAGES = json.loads(pages_env) if pages_env.startswith('[') else [p.strip() for p in pages_env.split(',') if p.strip()]
except:
    NOTION_ALLOWED_PAGES = []

print(f'Token: {"SET" if NOTION_API_TOKEN else "NOT SET"}')
print(f'Allowed Pages: {NOTION_ALLOWED_PAGES}')

In [2]:
def req(endpoint, payload):
    r = requests.post(f'{BASE_URL}{endpoint}', json=payload, headers={'Content-Type': 'application/json'}, timeout=5)
    print(f'Status: {r.status_code}')
    print(f'Response: {json.dumps(r.json() if r.text else {}, indent=2)}')
    return r

In [3]:
print('\n=== TEST 1: Health Check ===')
r = requests.get(f'{BASE_URL}/health')
print(f'Status: {r.status_code}')
print(json.dumps(r.json(), indent=2))


=== TEST 1: Health Check ===
Status: 200
{
  "app_env": "development"
}


In [4]:
print('\n=== TEST 2: Invalid Payload (expect 422) ===')
if NOTION_ALLOWED_PAGES:
    req('/notion-write/page', {'page_id': NOTION_ALLOWED_PAGES[0], 'updates': {'status': 'invalid'}})
else:
    print('No allowed pages')


=== TEST 2: Invalid Payload (expect 422) ===
Status: 422
Response: {
  "detail": "Invalid value 'invalid' for field 'status'"
}


In [5]:
print('\n=== TEST 3: Unauthorized Page (expect 403) ===')
req('/notion-write/page', {'page_id': 'unauthorized_xyz', 'updates': {'status': 'done'}})


=== TEST 3: Unauthorized Page (expect 403) ===
Status: 403
Response: {
  "detail": "Page 'unauthorized_xyz' not in allowlist"
}


<Response [403]>

In [6]:
print('\n=== TEST 4: Real Write (expect 200 or 502) ===')
if NOTION_ALLOWED_PAGES and NOTION_API_TOKEN:
    req('/notion-write/page', {'page_id': NOTION_ALLOWED_PAGES[0], 'updates': {'status': 'in_progress'}})
else:
    print(f'Token: {bool(NOTION_API_TOKEN)}, Pages: {bool(NOTION_ALLOWED_PAGES)}')


=== TEST 4: Real Write (expect 200 or 502) ===
Status: 502
Response: {
  "detail": "Notion API returned 400"
}
